# Chunking Strategy Comparison

Compares **5 chunking strategies** on 10 articles (11 once Modal Logic was added later for a formal/symbolic stress test) from the main naive-vs-rerank eval, so results stay comparable to that earlier work. Rather than running the full 20-question eval set through every strategy, testing used a hand-picked rotating subset (single-topic, cross-section composite, and one symbolic-content question) chosen to stress different failure modes rather than for full coverage.

**Scope, deliberately narrow:**
- Naive retrieval only (no reranking) -- isolates the chunking variable. Reranking was deliberately deferred until a chunking method was chosen; section-aware chunking is now the confirmed default (see `chunking_conclusions.md`), and reranking gets layered back on top as a separate follow-up comparison next.
- 5 strategies: 
 - section-aware (confirmed default), 
 - fixed-token-window (the "naive tutorial" baseline), 
 - paragraph-based, 
 - sentence-window, and 
 - semantic (embedding-based breakpoint detection, ruled out as a default due to high variance -- see conclusions).

Each strategy gets its own **namespace within a single shared index** (`sep-chunking-comparison`) so they don't interfere with your production index or with each other, without hitting Pinecone's 5-index cap on the free tier.

**Cost note on semantic chunking:** unlike the other four, semantic chunking embeds every individual *sentence* first (to compare them pairwise) before producing final chunks to embed. For a long article with 150+ sentences, that's 150+ extra embedding calls beyond what the other strategies need. Cheap in absolute dollar terms at `text-embedding-3-small` prices, but worth knowing before you scale this comparison up to more than 10 articles.

In [ ]:
# This notebook lives in notebooks/, one level below the project root
# where config.py, embeddings.py, vectorstore.py, etc. actually live.
# Jupyter's working directory defaults to wherever the .ipynb file sits,
# so without this, both `from config import settings`-style imports AND
# every relative file path later in this notebook (data/SEP.parquet,
# tests/eval_systematic.csv) would fail to find their targets.
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print("Working directory set to:", os.getcwd())


In [ ]:
import os
import re
import numpy as np
import pandas as pd
from uuid import uuid4
from tqdm import tqdm

from config import settings
from embeddings import get_embedder, get_embedding_dimension
from vectorstore import VectorDB
from chunker import tiktoken_len
from section_parser import split_into_sections
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

embed = get_embedder()
DIM = get_embedding_dimension()


## The five chunkers

Each takes a single article row (with `Title`, `Text`, `TOC`, etc.) and returns a list of `(section_label, chunk_text)` tuples -- matching the shape `ingest.py` already expects, so we can reuse the same metadata
and upsert logic for all five.

In [ ]:
def chunk_section_aware(row, chunk_size=400, chunk_overlap=20):
    """Current default: TOC-based section splitting, then token chunking within each section."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        length_function=tiktoken_len, separators=["\n\n", "\n", " ", ""]
    )
    sections, _ = split_into_sections(row["TOC"], row["Text"])
    out = []
    for section_title, section_text in sections:
        for chunk in splitter.split_text(section_text):
            out.append((section_title, chunk))
    return out


def chunk_fixed_window(row, chunk_size=400, chunk_overlap=20):
    """Baseline: ignore section structure entirely, chunk the whole article by token count."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        length_function=tiktoken_len, separators=["\n\n", "\n", " ", ""]
    )
    return [("(no section)", c) for c in splitter.split_text(row["Text"])]


def chunk_paragraph(row, target_tokens=400):
    """Split on blank-line paragraph breaks, then greedily merge paragraphs up to target_tokens."""
    paragraphs = [p.strip() for p in row["Text"].split("\n\n") if p.strip()]
    out, buf, buf_len = [], [], 0
    for p in paragraphs:
        p_len = tiktoken_len(p)
        if buf and buf_len + p_len > target_tokens:
            out.append(("(paragraph chunk)", " ".join(buf)))
            buf, buf_len = [], 0
        buf.append(p)
        buf_len += p_len
    if buf:
        out.append(("(paragraph chunk)", " ".join(buf)))
    return out


def chunk_sentence_window(row, sentences_per_chunk=6, overlap_sentences=1):
    """Group N sentences per chunk with a small overlap, ignoring section/paragraph structure."""
    sentences = re.split(r"(?<=[.!?])\s+", row["Text"])
    sentences = [s for s in sentences if s.strip()]
    out = []
    step = max(sentences_per_chunk - overlap_sentences, 1)
    for i in range(0, len(sentences), step):
        window = sentences[i:i + sentences_per_chunk]
        if window:
            out.append(("(sentence window)", " ".join(window)))
    return out


def chunk_semantic(row, breakpoint_percentile=95, max_chunk_tokens=800):
    """
    Embeds each sentence individually, finds semantic breakpoints via
    cosine distance between consecutive sentences (the classic
    percentile-threshold approach), and groups sentences into chunks at
    those breakpoints -- so a chunk boundary lands where the *topic*
    actually shifts, not at a fixed size or a structural marker.

    Cost note: embeds every sentence before producing final chunks --
    meaningfully more embedding calls per article than the other four
    strategies. See the notebook intro.

    max_chunk_tokens is a safety net: if a semantic segment between two
    breakpoints is unexpectedly huge (e.g. a long passage with no real
    topic shift), it still gets split further so nothing blows past a
    sane context-window size.
    """
    sentences = re.split(r"(?<=[.!?])\s+", row["Text"])
    sentences = [s.strip() for s in sentences if s.strip()]

    if len(sentences) < 3:
        return [("(semantic chunk)", row["Text"])]

    sentence_embeddings = np.array(embed.embed_documents(sentences))
    norms = np.linalg.norm(sentence_embeddings, axis=1, keepdims=True)
    normed = sentence_embeddings / norms
    sims = np.sum(normed[:-1] * normed[1:], axis=1)
    distances = 1 - sims  # cosine distance between each consecutive sentence pair

    threshold = np.percentile(distances, breakpoint_percentile)
    breakpoints = [i for i, d in enumerate(distances) if d > threshold]

    raw_chunks = []
    start = 0
    for bp in breakpoints:
        raw_chunks.append(" ".join(sentences[start:bp + 1]))
        start = bp + 1
    raw_chunks.append(" ".join(sentences[start:]))

    # Safety net: further split any oversized semantic segment by token count.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chunk_tokens, chunk_overlap=20,
        length_function=tiktoken_len, separators=["\n\n", "\n", " ", ""]
    )
    out = []
    for c in raw_chunks:
        if tiktoken_len(c) > max_chunk_tokens:
            for sub in splitter.split_text(c):
                out.append(("(semantic chunk, split)", sub))
        else:
            out.append(("(semantic chunk)", c))
    return out


CHUNKERS = {
    "section": chunk_section_aware,
    "fixed": chunk_fixed_window,
    "paragraph": chunk_paragraph,
    "sentence": chunk_sentence_window,
    "semantic": chunk_semantic,
}

## Load the same 10 articles used in the main eval

Adjust this list if your eval set's article titles differ slightly from
what's in `data/SEP.parquet`.


In [13]:
EVAL_ARTICLES = [
    "Peter Abelard", "Abduction", "The Concept of the Aesthetic",
    "Affirmative Action", "African Sage Philosophy", "Alienation",
    "al-Farabi", "Anselm of Canterbury", "Animalism",
    "Ancient Political Philosophy",
]

df = pd.read_parquet("SEP/data/SEP.parquet")
df = df[df["Title"].isin(EVAL_ARTICLES)].reset_index(drop=True)
print(f"Loaded {len(df)} of {len(EVAL_ARTICLES)} expected articles.")
missing = set(EVAL_ARTICLES) - set(df["Title"])
if missing:
    print("WARNING - not found in corpus:", missing)

Loaded 10 of 10 expected articles.


## Build one shared index, one namespace per strategy

Pinecone's free tier caps you at 5 serverless indexes *total*, project-wide -- which doesn't leave room for 5 experimental chunking indexes alongside your production index. The fix Pinecone itself recommends: **namespaces**. One index, five logically separate partitions -- same isolation between strategies, no risk of hitting the index cap again.


In [14]:
SHARED_INDEX_NAME = "sep-chunking-comparison"

def build_namespace_for_strategy(strategy_name, chunker_fn, df, index):
    texts, metadatas = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=strategy_name):
        chunks = chunker_fn(row)
        for i, (section_label, chunk_text) in enumerate(chunks):
            texts.append(chunk_text)
            metadatas.append({
                "title": row["Title"],
                "section": section_label,
                "chunk": i,
                "text": chunk_text,
                "strategy": strategy_name,
            })

    BATCH = 100
    for i in range(0, len(texts), BATCH):
        batch_texts = texts[i:i+BATCH]
        batch_meta = metadatas[i:i+BATCH]
        ids = [str(uuid4()) for _ in batch_texts]
        vectors = embed.embed_documents(batch_texts)
        # namespace= is what keeps each strategy's vectors logically
        # separate within the single shared index.
        index.upsert(vectors=zip(ids, vectors, batch_meta), namespace=strategy_name)

    print(f"{strategy_name}: {len(texts)} chunks indexed into namespace '{strategy_name}'")


vector_db = VectorDB()
vector_db.create_index(SHARED_INDEX_NAME, dimension=DIM, metric="cosine")
shared_index = vector_db.connect_to_index(SHARED_INDEX_NAME)

for name, fn in CHUNKERS.items():
    build_namespace_for_strategy(name, fn, df, shared_index)


Creating index 'sep-chunking-comparison' (dim=1536, metric=cosine)...
Index 'sep-chunking-comparison' created and ready.


section: 100%|██████████| 10/10 [00:00<00:00, 37.99it/s]


section: 502 chunks indexed into namespace 'section'


fixed: 100%|██████████| 10/10 [00:00<00:00, 38.09it/s]


fixed: 491 chunks indexed into namespace 'fixed'


...

paragraph: 445 chunks indexed into namespace 'paragraph'


sentence: 100%|██████████| 10/10 [00:00<00:00, 666.45it/s]


sentence: 691 chunks indexed into namespace 'sentence'


semantic: 100%|██████████| 10/10 [00:09<00:00,  1.03it/s]


semantic: 346 chunks indexed into namespace 'semantic'


## Query each strategy's namespace with the same eval questions

Naive retrieval only (`similarity_search`, no reranking) -- see the scoping note at the top.


In [34]:
llm = ChatOpenAI(
    openai_api_key=settings.OPENAI_API_KEY,
    model_name=settings.LLM_MODEL,
    temperature=settings.TEMPERATURE,
)

def generate_answer(question, docs):
    context = "\n\n".join(d.page_content for d in docs)
    prompt = (
        "Answer the question using only the context below. "
        "If the context doesn't contain the answer, say so -- don't make things up.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )
    return llm.invoke(prompt).content


def query_strategy(strategy_name, question, k=5):
    vectorstore = PineconeVectorStore(
        index=shared_index, embedding=embed, namespace=strategy_name
    )
    docs = vectorstore.similarity_search(question, k=k)
    return docs


QUESTIONS = [
    "Why does Abelard reject the theory that universals are real things?",
    "How does Abelard's theory of intentions determine moral worth?",
]

for question in QUESTIONS:
    print("=" * 100)
    print(f"QUESTION: {question}")
    print("=" * 100)

    for strategy_name in CHUNKERS:
        docs = query_strategy(strategy_name, question)
        print(f"\n--- {strategy_name} ---")
        for i, d in enumerate(docs, 1):
            title = d.metadata.get("title")
            section = d.metadata.get("section")
            snippet = d.page_content[:400].replace("\n", " ")
            print(f"[{i}] ({title} | {section}) {snippet}")

        answer = generate_answer(question, docs)
        print(f"\nGENERATED ANSWER ({strategy_name}):\n{answer}")
    print()

QUESTION: Why does Abelard reject the theory that universals are real things?

--- section ---
[1] (Peter Abelard | Metaphysics) If we object to this last piece of reasoning, on the grounds that individuals are unique in virtue of their non-essential features, Abelard replies that this view “makes accidents prior to substance.” That is, the objection claims that individual things are individual in virtue of features that contingently characterize them, which confuses things with their features.  Prospects are no better for 
[2] (Peter Abelard | Metaphysics) 2. Metaphysics  Abelard’s metaphysics is the first great example of nominalism in the Western tradition. While his view that universals are mere words (nomina) justifies the label, nominalism—or, better, irrealism—is the hallmark of Abelard’s entire metaphysics. He is an irrealist not only about universals, but also about propositions, events, times other than the present, natural kinds, relations
[3] (Peter Abelard | Metaphysics) A

#### Observations: Abelard questions above (retrieval + generation)

**Q1 (universals) and Q2 (intentions), both on Peter Abelard.**

- **All five strategies now generate correct, complete answers on both questions** -- this is the one pair where generation didn't introduce new problems. `fixed`, `paragraph`, `sentence`, and `semantic` all independently reconstructed the "Socrates and an ass, both wholly animal" contradiction argument in their answers, even when it wasn't fully visible in the printed retrieved snippets -- meaning it wax present in the full chunk text the model saw, just truncated in what
  we're printing.

- **`section`'s generated answer was the thinnest of the five**, despite retrieving nearly the same chunks as `fixed`/`paragraph`. It correctly answered the question but omitted the specific contradiction argument the other four included. Same underlying evidence, less complete synthesis -- a small but real gap that wasn't visible from the retrieval-only comparison.

- **`semantic` remains the volatile one, but less dramatically than the retrieval-only read suggested.** On Q1 it retrieved chunks that included an off-topic Theology/heresy passage, yet the generate answer still came out accurate and complete -- the model apparently filtered past the irrelevant chunk on its own. On Q2 it again produced a full, correct answer.

**Revised takeaway:** on single-topic questions with an unambiguous answer clearly stated in the source, generation is fairly robust to retrieval noise -- a stray off-topic chunk doesn't necessarily wreck the answer if the right chunk is also present. The retrieval-only comparison slightly overstated how much these differences matter once generation is in the loop, except for `section`'s modest completeness gap, which is new information this batch surfaced.

In [33]:
llm = ChatOpenAI(
    openai_api_key=settings.OPENAI_API_KEY,
    model_name=settings.LLM_MODEL,
    temperature=settings.TEMPERATURE,
)

def generate_answer(question, docs):
    context = "\n\n".join(d.page_content for d in docs)
    prompt = (
        "Answer the question using only the context below. "
        "If the context doesn't contain the answer, say so -- don't make things up.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )
    return llm.invoke(prompt).content


def query_strategy(strategy_name, question, k=5):
    vectorstore = PineconeVectorStore(
        index=shared_index, embedding=embed, namespace=strategy_name
    )
    docs = vectorstore.similarity_search(question, k=k)
    return docs


QUESTIONS = [
    # Abelard: early biography (section 1) + late theology (section 7)
    "How did Abelard's conflict with Bernard of Clairvaux relate to his theory of identity used to explain the Trinity?",
    # Anselm: ontological argument (section 2) + ethics of freedom (section 4)
    "How does Anselm's definition of God as 'that than which nothing greater can be thought' connect to his account of freedom and sin?",
    # Ancient Political Philosophy: Socrates' trial (section 3) + Cicero (section 6)
    "How does Cicero's res publica compare to Socrates' relationship to the Laws of Athens at his trial?",
]

for question in QUESTIONS:
    print("=" * 100)
    print(f"QUESTION: {question}")
    print("=" * 100)

    for strategy_name in CHUNKERS:
        docs = query_strategy(strategy_name, question)
        print(f"\n--- {strategy_name} ---")
        for i, d in enumerate(docs, 1):
            title = d.metadata.get("title")
            section = d.metadata.get("section")
            snippet = d.page_content[:400].replace("\n", " ")
            print(f"[{i}] ({title} | {section}) {snippet}")

        answer = generate_answer(question, docs)
        print(f"\nGENERATED ANSWER ({strategy_name}):\n{answer}")
    print()

QUESTION: How did Abelard's conflict with Bernard of Clairvaux relate to his theory of identity used to explain the Trinity?

--- section ---
[1] (Peter Abelard | Theology) Abelard’s arguments for rejecting (a)–(c) are sophisticated and subtle. For the claim that reason may be fruitfully applied to a particular article of faith, Abelard offers a particular case study in his own writings. The bulk of Abelard’s work on theology is devoted to his dialectical investigation of the Trinity. He elaborates an original theory of identity to address issues surrounding the Trin
[2] (Peter Abelard | Theology) Now for the payoff. Abelard deploys his theory of identity to shed light on the Trinity as follows. The three Persons are essentially the same as one another, since they are all the same concrete thing (namely God). They differ from one another in definition, since what it is to be the Father is not the same as what it is to be the Son or what it is to be the Holy Spirit. The three Persons ar

#### Observations: cross-section stress questions (retrieval + generation)

Testing whether the full pipeline, not just retrieval, can handle
questions requiring content from two distant, non-adjacent parts of
the same article.

- **Q1 (Abelard: Bernard conflict + Trinity identity)** -- still not a fair test; both halves live in the same section (7. Theology). Useful as a control instead: four of five strategies (`section`, `fixed`, `paragraph`, `sentence`) explicitly declined to connect the two ideas even though both were sitting in the same retrieved chunk set, right next to each other. Only `semantic`'s generated answer attempted a real connection, and even that stayed shallow.

- **Q2 (Anselm: ontological argument + freedom/sin)** -- this is where the earlier retrieval-only note needs correcting. At the chunk level, `section` looked like a partial win (it retrieved a real mix of both topics). At the generation level, `section`, `fixed`, and `paragraph` all produced the *same* refusal anyway ("discussed separately... not directly connected"), despite having the material to connect. Only `semantic` generated an actual synthesized answer linking Anselm's God-definition to his account of freedom. Retrieval succeeding did not translate into a synthesized answer for 4 of 5 strategies.

- **Q3 (Cicero res publica vs. Socrates' trial)** -- confirmed and sharpened. All five strategies refused to answer. The new detail: `paragraph` actually retrieved the correct res publica definition passage in its top 5 this time, and *still* declined to connect it to Socrates. Having the right chunk in context wasn't enough on its own.

**Revised hypothesis, updated from the retrieval-only version:** retrieval and generation are two separate failure points, and generation refusal is the more common and more decisive one of the two. Several strategies retrieved adequate evidence for both halves of a composite question and the model still declined to synthesize across it. This is stronger, more direct evidence for the multiquery/agentic next step than the retrieval-only data alone -- the problem isn't only "find both halves," it's "be willing to connect them once found." `semantic` was the one strategy that attempted synthesis somewhat consistently across this batch, which is worth another look, though still not enough data to trust it as a default given its volatility elsewhere.

In [30]:
modal_df = pd.read_parquet("SEP/data/SEP.parquet")
modal_df = modal_df[modal_df["Title"] == "Modal Logic"].reset_index(drop=True)
print(f"Found {len(modal_df)} row(s) for Modal Logic")

for name, fn in CHUNKERS.items():
    build_namespace_for_strategy(name, fn, modal_df, shared_index)

Found 1 row(s) for Modal Logic


section: 100%|██████████| 1/1 [00:00<00:00,  8.62it/s]


section: 75 chunks indexed into namespace 'section'


fixed: 100%|██████████| 1/1 [00:00<00:00, 22.22it/s]


fixed: 72 chunks indexed into namespace 'fixed'


...

paragraph: 69 chunks indexed into namespace 'paragraph'


sentence: 100%|██████████| 1/1 [00:00<00:00, 250.00it/s]


sentence: 122 chunks indexed into namespace 'sentence'


semantic: 100%|██████████| 1/1 [00:01<00:00,  1.67s/it]


semantic: 55 chunks indexed into namespace 'semantic'


In [32]:
llm = ChatOpenAI(
    openai_api_key=settings.OPENAI_API_KEY,
    model_name=settings.LLM_MODEL,
    temperature=settings.TEMPERATURE,
)

def generate_answer(question, docs):
    context = "\n\n".join(d.page_content for d in docs)
    prompt = (
        "Answer the question using only the context below. "
        "If the context doesn't contain the answer, say so -- don't make things up.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )
    return llm.invoke(prompt).content


QUESTIONS = [
    # Ancient Political Philosophy -- deliberately distant sections
    "How does the Greek sophist debate over nomos and phusis compare to the Stoic conception of natural law?",
    "How does Aristotle's account of the many's collective judgment compare to Cicero's mixed constitution?",
    "How does Plato's account of justice in the Republic compare to Seneca's treatment of mercy under the Roman Empire?",
    # Modal Logic -- formal/symbolic content, structural stress test
    "What is the difference between the axioms of system S4 and system S5?",
]

for question in QUESTIONS:
    print("=" * 100)
    print(f"QUESTION: {question}")
    print("=" * 100)

    for strategy_name in CHUNKERS:
        docs = query_strategy(strategy_name, question)
        print(f"\n--- {strategy_name} ---")
        for i, d in enumerate(docs, 1):
            title = d.metadata.get("title")
            section = d.metadata.get("section")
            snippet = d.page_content[:250].replace("\n", " ")
            print(f"[{i}] ({title} | {section}) {snippet}")

        answer = generate_answer(question, docs)
        print(f"\nGENERATED ANSWER ({strategy_name}):\n{answer}")
    print()

QUESTION: How does the Greek sophist debate over nomos and phusis compare to the Stoic conception of natural law?

--- section ---
[1] (Ancient Political Philosophy | Politics and Philosophy in Ancient Greece) and practical abilities; whereas for Thrasymachus (an ambassador to Athens from Chalcedon, a city in Asia Minor near modern Istanbul, who is depicted in Plato’s Republic), it was a cause of condemnation, the powerful in any city imposing laws to serv
[2] (Ancient Political Philosophy | Politics and Philosophy in Ancient Greece) Most of those generally recognized as “wise men” (sophoi) and “students of nature” (physikoi) who appeared in this milieu, thought within the same broad terms as the poets and orators. Justice was widely, if not universally, treated as a fundamental 
[3] (Ancient Political Philosophy | Hellenistic Philosophies and Politics) with the cosmos, has much exercised both Stoic thinkers and subsequent interpreters. And which law? How does or should the law of a pa

#### Observations: cross-section synthesis questions + Modal Logic stress test

**Three composite Ancient Political Philosophy questions** (nomos/phusis vs. Stoic law; Aristotle's many vs. Cicero's mixed constitution; Plato's justice vs. Seneca's mercy) - the dominant result was refusal, not retrieval failure. `section`, `fixed`, `sentence`, and `semantic` mostly answered "the context does not provide a direct comparison" even when their own retrieved chunks contained clear evidence for *both* halves of the question, sitting side by side in context.

**This reframes the earlier hypothesis.** It's not just that a single embedding struggles to retrieve both halves of an imbalanced question (true on some questions, e.g. the earlier Cicero-vs-Socrates test) - here, several strategies retrieved good evidence for both sides and *still* the generator declined to synthesize across them. That's a distinct, additional failure mode on top of the retrieval one: given adjacent-but-separate evidence, the model is cautious about connecting it into an explicit comparison the source text doesn't state directly.

**`paragraph` and `sentence` were the only strategies that reliably attempted real synthesis** on Q1, producing accurate, well-reasoned comparisons - while `section` and `fixed` retrieved nearly identical chunks on the same question and still refused. Same underlying information, different chunk boundaries, different generation behavior. Worth flagging as evidence that exact chunk framing (not just chunk content) can tip a model between synthesizing and refusing.

**Modal Logic (formal/symbolic stress test, not in main eval set):** clean result across all five strategies - no chunker mangled LaTeX or split a formula mid-symbol, all five retrieved the correct S4/S5 axiom passages, all five generated accurate answers. `sentence` gave the most technically precise answer, correctly naming axioms (4) and (5) by their frame-condition names (transitive, Euclidean). Chunking robustness on symbolic content is not a live concern based on this test.

**Combined implication:** confirms multiquery decomposition is likely necessary (per the earlier hypothesis), but adds a second target for the future agentic layer - an explicit "synthesize across retrieved evidence" step or prompt adjustment may matter as much as retrieving the right chunks in the first place. Retrieval quality and generation willingness-to-synthesize are separate variables, and this batch is the first evidence that both need addressing, not just retrieval.

## Wrapping The Notebook: What I Actually Learned

I went into this wanting a clean answer: which of five chunking strategies retrieves best? Section-aware, fixed-window, paragraph, sentence-window, semantic. Ten articles, a handful of single-topic questions, then some harder composite ones, then one formal/symbolic article just to see if anything broke.

I didn't get a clean winner. For a while that felt like a wasted afternoon. It wasn't, and here's why.

### What didn't happen

No strategy pulled decisively ahead on straightforward, single-topic questions. Section-aware, fixed-window, and paragraph-based chunking retrieved nearly identical chunks on most of the questions I tried, word for word in places. When I ran those through generation too, section-aware's answers came out slightly thinner than the others on the same underlying evidence, small enough to be noise but worth flagging rather than smoothing over. That's not what I expected. Section-aware chunking was built on the assumption that respecting an article's real structure (via its table of contents) would beat blindly splitting by token count. On this corpus, that assumption mostly didn't pay off. The paragraph breaks in a well-written SEP article already line up with the topic breaks closely enough that a dumb splitter gets almost the same result as a structure-aware one.

### What did happen, decisively

**Semantic chunking is out.** It was the only strategy that ever clearly won on a question (catching a specific detail, like Abelard's sleeping-monk example, that nothing else found) and the only strategy that ever clearly lost (drifting into an unrelated passage about heresy when asked about universals). That volatility is itself the finding. I don't want my retrieval quality depending on a coin flip, so semantic chunking is disqualified as a default even though it has a real, occasionally impressive ceiling.

**The chunking layer isn't where the real problem lives.** This is the one that actually matters. When I pushed into harder composite questions (comparing something from one part of an article to something from another part entirely), nearly every strategy failed the same way: not by failing to retrieve the relevant content, but by retrieving it and then refusing to synthesize it into an actual comparison. The evidence for both halves of the question was sitting right there in the context window, and the model still said "the context doesn't provide a direct comparison." That's not a retrieval problem. That's a generation problem, and no amount of re-chunking the source text was going to fix it.

Put together with what I found yesterday (the reranker sometimes discarding the one chunk that actually contained the answer, even when a wider net had caught it), a pattern is forming. The bottleneck in this pipeline isn't how the text gets cut up. It's what happens after retrieval: which chunks survive the narrowing step, and whether the model is willing to reason across what it's given rather than just paraphrasing whichever chunk looks most directly on-topic.

### The decision

I'm keeping section-aware chunking as the default. Not because it won the comparison (it didn't, it mostly tied), but because nothing beat it, it matches the real structure of the source articles (which matters for how I explain this project later, even where it didn't move the numbers), and switching to a roughly-tied alternative isn't worth the churn. Semantic chunking is ruled out for anything that needs to run unattended, given how unpredictable it was. One thing worth a second look later: semantic was also the strategy most willing to attempt real synthesis on the composite questions, when the other four mostly just refused. That doesn't undo the volatility problem, but it's a data point in its favor I don't want to lose track of.

### Where the effort goes next

Not into a sixth chunking strategy. Into the generation layer. The plan was always to build a multiquery and self-critique step eventually, and this experiment gave me a real reason to move that up rather than treat it as a someday-project: it's the actual bottleneck, demonstrated with real questions and real failures, not a guess.

One cheap thing worth trying before building all of that: a small prompt tweak that explicitly invites the model to connect evidence across chunks rather than requiring the source to state the comparison outright. If that alone fixes some of these refusals, it tells me part of this is a prompt problem, not purely an architecture problem, and it's worth ruling that out before investing real time in the bigger agentic build.

### Final words

A little deflated that there's no single "winner" chunker to point to. But looking back at it straight: I ran a real test, on real data, with a hypothesis stated in advance, and the result was "this variable matters less than I thought, here's the one that matters more." That's a better outcome than confirming what I already believed going in would have been. It's just less satisfying to write up.